# Stress Prediction — 누적 피처 제거 (다른 팀 힌트 기반)

다른 팀 힌트: "피처를 많이 추가한다고 항상 좋아지지 않음", "중복되거나 불안정한 피처
제거가 더 효과적".

오늘 LOO(Leave-One-Out) 실험에서 하나씩 뺐을 때 delta가 **음수**(=빼면 좋아짐)였던
피처가 13개 있었습니다. 각각은 노이즈 수준이었지만, **한꺼번에 여러 개를 빼면 누적되어
진짜 신호가 될 수 있다**는 가설을 테스트합니다.

LOO 결과 순위(delta 오름차순, 음수만): gender, medical_history, bmi,
cholesterol_glucose_ratio, activity, missing_count, sleep_pattern, height,
smoke_status, weight, mean_arterial_pressure, systolic_blood_pressure, cholesterol


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 800  # 탐색용
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]

# LOO 실험에서 delta가 음수였던 순서 (효과 큰 순)
NEGATIVE_DELTA_COLUMNS_RANKED = [
    "gender", "medical_history", "bmi", "cholesterol_glucose_ratio", "activity",
    "missing_count", "sleep_pattern", "height", "smoke_status", "weight",
    "mean_arterial_pressure", "systolic_blood_pressure", "cholesterol",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## CV MAE 계산 함수 (pair는 컬럼 제거와 무관하므로 fold당 한 번만 계산)

주의: `bmi`, `cholesterol_glucose_ratio`, `height`, `weight`, `cholesterol`은
`PAIR_COLUMNS`에도 쓰이는 컬럼입니다. 여기서는 **tree 모델 입력에서만** 제거하고,
pair 방식은 원래 8개 컬럼 그대로 유지합니다 (pair까지 건드리면 28조합 앙상블이 깨짐).


In [2]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)

pair_cache = []
for train_idx, val_idx in cv.split(raw_X):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    pred_pair = pair_prediction(train_feat, val_feat, y_train)
    pair_cache.append((train_idx, val_idx, pred_pair, y_train))
print("pair 캐시 완료")


def cv_mae_dropping(drop_cols):
    oof = np.zeros(len(y))
    for train_idx, val_idx, pred_pair, y_train in pair_cache:
        train_feat = raw_X.iloc[train_idx].reset_index(drop=True).drop(columns=drop_cols, errors="ignore")
        val_feat = raw_X.iloc[val_idx].reset_index(drop=True).drop(columns=drop_cols, errors="ignore")
        pred_tree = tree_prediction(train_feat, val_feat, y_train)
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        oof[val_idx] = combined
    return mean_absolute_error(y, oof)


pair 캐시 완료


## 누적 제거 테스트 (상위 1개 -> 3개 -> 5개 -> 7개 -> 10개 -> 13개 전부)

In [3]:

baseline_mae = cv_mae_dropping([])
print(f"### baseline(아무것도 제거 안 함) CV MAE: {baseline_mae:.6f}")
print()

rows = [{"n_removed": 0, "removed_columns": "(none)", "cv_mae": baseline_mae}]
for n in [1, 3, 5, 7, 10, 13]:
    drop_cols = NEGATIVE_DELTA_COLUMNS_RANKED[:n]
    mae = cv_mae_dropping(drop_cols)
    rows.append({"n_removed": n, "removed_columns": ", ".join(drop_cols), "cv_mae": mae})
    print(f"상위 {n}개 제거 {drop_cols}: {mae:.6f} (delta {mae - baseline_mae:+.6f})")

result_df = pd.DataFrame(rows)
print()
display(result_df)
print()
print("(참고) v34 원본(피처 제거 없음) 실제 CV MAE: 0.148033")


### baseline(아무것도 제거 안 함) CV MAE: 0.147993

상위 1개 제거 ['gender']: 0.147697 (delta -0.000297)
상위 3개 제거 ['gender', 'medical_history', 'bmi']: 0.148567 (delta +0.000573)
상위 5개 제거 ['gender', 'medical_history', 'bmi', 'cholesterol_glucose_ratio', 'activity']: 0.148227 (delta +0.000233)
상위 7개 제거 ['gender', 'medical_history', 'bmi', 'cholesterol_glucose_ratio', 'activity', 'missing_count', 'sleep_pattern']: 0.147720 (delta -0.000273)
상위 10개 제거 ['gender', 'medical_history', 'bmi', 'cholesterol_glucose_ratio', 'activity', 'missing_count', 'sleep_pattern', 'height', 'smoke_status', 'weight']: 0.149947 (delta +0.001953)
상위 13개 제거 ['gender', 'medical_history', 'bmi', 'cholesterol_glucose_ratio', 'activity', 'missing_count', 'sleep_pattern', 'height', 'smoke_status', 'weight', 'mean_arterial_pressure', 'systolic_blood_pressure', 'cholesterol']: 0.156530 (delta +0.008537)



,n_removed,removed_columns,cv_mae
0,0,(none),0.147993
1,1,gender,0.147697
2,3,"gender, medical_history, bmi",0.148567
3,5,"gender, medical_history, bmi, cholesterol_gluc...",0.148227
4,7,"gender, medical_history, bmi, cholesterol_gluc...",0.147720
5,10,"gender, medical_history, bmi, cholesterol_gluc...",0.149947
6,13,"gender, medical_history, bmi, cholesterol_gluc...",0.156530



(참고) v34 원본(피처 제거 없음) 실제 CV MAE: 0.148033


## 최종 제출 파일 생성

가장 낮은 `cv_mae`를 준 `n_removed` 값으로 `BEST_N_REMOVED`를 수정하세요.
0보다 확실히 낮은 지점이 없으면 실행하지 말고 v34를 유지하세요.


In [5]:

BEST_N_REMOVED = 1  # 결과 보고 수정 (0 = 제거 안 함 = v34 원본과 동일)
FINAL_N_ESTIMATORS = 1200

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

drop_cols = NEGATIVE_DELTA_COLUMNS_RANKED[:BEST_N_REMOVED]
N_ESTIMATORS = FINAL_N_ESTIMATORS

final_train_feat = raw_X.drop(columns=drop_cols, errors="ignore")
final_test_feat = raw_X_test.drop(columns=drop_cols, errors="ignore")

final_pair = pair_prediction(raw_X, raw_X_test, y)  # pair는 항상 원본 8개 컬럼 사용
final_tree = tree_prediction(final_train_feat, final_test_feat, y)
final_prediction = np.clip(np.round((1 - PAIR_WEIGHT) * final_tree + PAIR_WEIGHT * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_cumulative_drop_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("제거한 컬럼:", drop_cols)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_cumulative_drop_0806.csv
제거한 컬럼: ['gender']
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.57
9,TEST_0009,0.88
